In [32]:
# Install the requests library if not already installed
!pip install requests --quiet

In [34]:
import os, shutil, time

# Create an "original" piece of evidence
with open("evidence_original.txt", "w") as f:
    f.write("Case File #001 - Suspect device log")
orig_stat = os.stat("evidence_original.txt")
print("Original file - created (ctime):", time.ctime(orig_stat.st_ctime))
print("Original file - modified (mtime):", time.ctime(orig_stat.st_mtime))

# Simulate an investigator copying the file to a workstation
time.sleep(1)
shutil.copy2("evidence_original.txt", "evidence_copy.txt")
copy_stat = os.stat("evidence_copy.txt")
print("\nCopied file - created (ctime):", time.ctime(copy_stat.st_ctime))
print("Copied file - modified (mtime):", time.ctime(copy_stat.st_mtime))

print("\nLocard's Exchange Principle: the copy operation itself created a new")
print("ctime on the destination file - every interaction leaves a trace.")

Original file - created (ctime): Tue Aug  4 04:13:33 2026
Original file - modified (mtime): Tue Aug  4 04:13:33 2026

Copied file - created (ctime): Tue Aug  4 04:13:34 2026
Copied file - modified (mtime): Tue Aug  4 04:13:33 2026

Locard's Exchange Principle: the copy operation itself created a new
ctime on the destination file - every interaction leaves a trace.


In [35]:
import hashlib, datetime
def sha256_of(filename):
    with open(filename, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

# Step 1: Investigator seizes evidence and hashes it immediately
with open("evidence.txt", "w") as f:
    f.write("Suspect chat log: meeting at 10pm, bring the drive.")
seizure_hash = sha256_of("evidence.txt")
custody_log = []
custody_log.append(f"{datetime.datetime.now()} - SEIZED by Officer A - hash={seizure_hash}")

# Step 2: Evidence is later handed to a forensic analyst; verify integrity
handoff_hash = sha256_of("evidence.txt")
if handoff_hash == seizure_hash:
    custody_log.append(f"{datetime.datetime.now()} - RECEIVED by Analyst B - integrity VERIFIED")
else:
    custody_log.append(f"{datetime.datetime.now()} - RECEIVED by Analyst B - integrity FAILED (tampered!)")

print("=== Chain of Custody Log ===")
for entry in custody_log:
    print(entry)

=== Chain of Custody Log ===
2026-08-04 04:13:48.777851 - SEIZED by Officer A - hash=b7eb05cf41efbcc50adc36b12679ce4f12de58460e59500eb6bb62858810f84c
2026-08-04 04:13:48.778191 - RECEIVED by Analyst B - integrity VERIFIED


In [37]:
import re
def check_email(sender, subject, body):
    flags = []
    if re.search(r"(urgent|verify your account|suspended|click here)", body,
                 re.IGNORECASE):
        flags.append("Urgency/pressure language detected")
    if re.search(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", body):
        flags.append("Raw IP address link found in body")
    domain = sender.split("@")[-1]
    if any(k in domain.lower() for k in ["secure","verify","update"]) and "@gmail" not \
        in sender:
        flags.append("Suspicious sender domain naming pattern")
    return flags
emails = [
    ("support@paypal.com", "Your monthly statement", "Please find your statement attached."),
    ("alert@paypal-secure-verify.com", "URGENT: Verify your account", "Click here: http://192.168.10.5/login"),
]
for sender, subject, body in emails:
    issues = check_email(sender, subject, body)
    verdict = "PHISHING SUSPECTED" if issues else "Looks legitimate"
    print(f"\nFrom: {sender}\nSubject: {subject}\nVerdict: {verdict}")
    for i in issues:
        print(" -", i)


From: support@paypal.com
Subject: Your monthly statement
Verdict: Looks legitimate

From: alert@paypal-secure-verify.com
Subject: URGENT: Verify your account
Verdict: PHISHING SUSPECTED
 - Urgency/pressure language detected
 - Raw IP address link found in body
 - Suspicious sender domain naming pattern


In [38]:
import socket
domains = ["www.google.com", "www.python.org", "notarealdomain12345.com"]
print("OSINT Domain Reconnaissance")
for d in domains:
    try:
        ip = socket.gethostbyname(d)
        print(f"{d:30s} -> {ip}")
    except socket.gaierror:
        print(f"{d:30s} -> Could not resolve (invalid/unreachable)")

OSINT Domain Reconnaissance
www.google.com                 -> 142.251.152.119
www.python.org                 -> 151.101.0.223
notarealdomain12345.com        -> Could not resolve (invalid/unreachable)


In [39]:
import datetime
def investigation_report(case_id, evidence_list):
    stages = {
        "Identification": f"Incident reported for case {case_id}. Devices/logs identified for review.",
        "Collection": f"{len(evidence_list)} item(s) collected: {', '.join(evidence_list)}",
        "Preservation": "All items hashed (SHA-256) and stored in a write-protected evidence folder.",
        "Analysis": "Log files and file metadata examined for indicators of compromise.",
        "Reporting": "Findings compiled into a structured forensic report for legal review.",
    }
    print(f"=== Investigation Lifecycle: Case {case_id} ===")
    print(f"Generated: {datetime.datetime.now()}\n")
    for stage, detail in stages.items():
        print(f"[{stage}]")
        print(f" {detail}\n")
investigation_report("CASE-2026-014", ["laptop_disk_image.dd", "router_traffic.pcap", "email_headers.txt"])

=== Investigation Lifecycle: Case CASE-2026-014 ===
Generated: 2026-08-04 04:14:21.569438

[Identification]
 Incident reported for case CASE-2026-014. Devices/logs identified for review.

[Collection]
 3 item(s) collected: laptop_disk_image.dd, router_traffic.pcap, email_headers.txt

[Preservation]
 All items hashed (SHA-256) and stored in a write-protected evidence folder.

[Analysis]
 Log files and file metadata examined for indicators of compromise.

[Reporting]
 Findings compiled into a structured forensic report for legal review.



In [40]:
import hashlib
def sha256_of(filename):
    with open(filename, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()
# Create a sample "disk" file (simulating a small storage device)
with open("original_disk.img", "wb") as f:
    f.write(b"HEADER" + bytes(range(256)) * 4 + b"FOOTER")
# Step 1: Create a bit-for-bit forensic copy (bit-stream image)
with open("original_disk.img", "rb") as src, open("forensic_copy.img", "wb") as dst:
    dst.write(src.read())
# Step 2: Verify the copy is identical using hashing
original_hash = sha256_of("original_disk.img")
copy_hash = sha256_of("forensic_copy.img")
print("Original image hash:", original_hash)
print("Forensic copy hash :", copy_hash)
print("Match:", "VERIFIED - exact bit-stream copy" if original_hash == copy_hash else "MISMATCH - copy corrupted")

Original image hash: d50630bee2bf9926344781042d10cb09377730818efda8e80f9eeec21008a9b0
Forensic copy hash : d50630bee2bf9926344781042d10cb09377730818efda8e80f9eeec21008a9b0
Match: VERIFIED - exact bit-stream copy


In [41]:
SIGNATURES = {
    b"\xFF\xD8\xFF": "JPEG image",
    b"\x89PNG": "PNG image",
    b"%PDF": "PDF document",
    b"PK\x03\x04": "ZIP archive (or .docx/.xlsx)",
}

def identify_file(path):
    with open(path, "rb") as f:
        header = f.read(8)
        for sig, filetype in SIGNATURES.items():
            if header.startswith(sig):
                return filetype
    return "Unknown file type"

# Create sample files with fake/renamed extensions to test signature detection
with open("photo.txt", "wb") as f: # renamed JPEG
    f.write(b"\xFF\xD8\xFF\xE0" + b"\x00" * 20)

with open("document.dat", "wb") as f: # renamed PDF
    f.write(b"%PDF-1.4" + b"\x00" * 20)

for filename in ["photo.txt", "document.dat"]:
    print(f"{filename:15s} -> Actual type: {identify_file(filename)}")

photo.txt       -> Actual type: JPEG image
document.dat    -> Actual type: PDF document


In [42]:
import os
# Step 1: Create a file, then "delete" it (simulating accidental/malicious deletion)
with open("secret_note.txt", "w") as f:
    f.write("MEETING_POINT: warehouse 7, 11pm")
with open("secret_note.txt", "rb") as f:
    original_bytes = f.read()
os.remove("secret_note.txt")
print("File 'secret_note.txt' deleted.")
print("Exists on disk now?", os.path.exists("secret_note.txt"))
# Step 2: Simulate raw disk scanning - in real forensics, tools like Autopsy
# scan unallocated space for byte patterns. Here we search a "disk buffer"
# (which still holds the bytes) for the recoverable content.
disk_buffer = original_bytes + b"\x00" * 50 # unallocated space padding
marker = b"MEETING_POINT"
index = disk_buffer.find(marker)
if index != -1:
    recovered = disk_buffer[index:].split(b"\x00")[0]
    with open("recovered_note.txt", "wb") as f:
        f.write(recovered)
    print("Recovered content:", recovered.decode())
else:
    print("No recoverable data found.")

File 'secret_note.txt' deleted.
Exists on disk now? False
Recovered content: MEETING_POINT: warehouse 7, 11pm


In [43]:
import re
from collections import Counter

# Create the sample system log so the script runs standalone
with open("login_attempts.log", "w") as f:
    f.write("2026-07-08 09:00:01 LOGIN SUCCESS user=alice ip=10.0.0.5\n")
    f.write("2026-07-08 09:01:15 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:20 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:25 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:30 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:35 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:02:00 LOGIN SUCCESS user=bob ip=10.0.0.8\n")

failed_by_ip = Counter()
with open("login_attempts.log") as f:
    for line in f:
        if "LOGIN FAILED" in line:
            match = re.search(r"ip=(\S+)", line)
            if match:
                failed_by_ip[match.group(1)] += 1

print("System log forensic analysis - failed logins by source IP:")
for ip, count in failed_by_ip.items():
    print(f" {ip}: {count} failed attempts")
    if count >= 5:
        print(f" -> Evidence of brute-force intrusion attempt from {ip}")

System log forensic analysis - failed logins by source IP:
 203.0.113.99: 5 failed attempts
 -> Evidence of brute-force intrusion attempt from 203.0.113.99


In [44]:
raw_header = """Delivered-To: victim@example.com
Received: from mail-relay-99.suspicious-host.ru (unknown [45.33.32.156])
by mx.example.com; Tue, 08 Jul 2026 09:15:00 +0000
From: "Bank Support" <support@paypal-secure-verify.com>
Reply-To: attacker@totallynotscam.net
Subject: Urgent: Verify your account
"""
def analyze_header(header_text):
    findings = []
    for line in header_text.splitlines():
        if line.startswith("Received:") and "suspicious" in line.lower():
            findings.append(f"Suspicious relay server found: {line.strip()}")
        if line.startswith("Reply-To:"):
            findings.append(f"Reply-To differs from sender - possible spoofing:\n{line.strip()}")
        if line.startswith("From:") and "-secure-verify" in line:
            findings.append(f"Sender domain uses suspicious naming: {line.strip()}")
    return findings

print("Email Header Forensic Analysis")
for finding in analyze_header(raw_header):
    print("-", finding)

Email Header Forensic Analysis
- Suspicious relay server found: Received: from mail-relay-99.suspicious-host.ru (unknown [45.33.32.156])
- Sender domain uses suspicious naming: From: "Bank Support" <support@paypal-secure-verify.com>
- Reply-To differs from sender - possible spoofing:
Reply-To: attacker@totallynotscam.net


In [45]:
# Create a sample simulated packet log so this runs standalone
with open("traffic.log", "w") as f:
    f.write("10.0.0.9 -> 10.0.0.5:22 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:23 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:25 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:80 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:443 SYN\n")
    f.write("10.0.0.20 -> 10.0.0.5:80 SYN\n")
    f.write("10.0.0.20 -> 10.0.0.5:80 ACK\n")

# Count distinct destination ports probed by each source IP
ports_by_source = {}
with open("traffic.log") as f:
    for line in f:
        parts = line.split()
        src = parts[0]
        dst_port = parts[2].split(":")[1]
        ports_by_source.setdefault(src, set()).add(dst_port)

print("Network traffic forensic analysis:")
for src, ports in ports_by_source.items():
    print(f" {src}: contacted {len(ports)} distinct port(s) -> {sorted(ports)}")
    if len(ports) >= 4:
        print(f" -> ALERT: {src} shows port-scanning behaviour")

Network traffic forensic analysis:
 10.0.0.9: contacted 5 distinct port(s) -> ['22', '23', '25', '443', '80']
 -> ALERT: 10.0.0.9 shows port-scanning behaviour
 10.0.0.20: contacted 1 distinct port(s) -> ['80']


In [46]:
from PIL import Image
from PIL.ExifTags import TAGS
# Install once: pip install Pillow
def create_sample_image(path):
    # Creates a small test image (a real photo would already have EXIF data)
    img = Image.new("RGB", (100, 100), color="blue")
    img.save(path)
def show_metadata(path):
    img = Image.open(path)
    print("File:", path)
    print("Format:", img.format)
    print("Size:", img.size)
    print("Mode:", img.mode)
    exif_data = img._getexif() if hasattr(img, "_getexif") else None
    if exif_data:
        for tag_id, value in exif_data.items():
            tag = TAGS.get(tag_id, tag_id)
            print(f" {tag}: {value}")
    else:
        print(" No EXIF metadata found (this sample image has none).")
        print(" Real photos from phones/cameras typically contain GPS,")
        print(" device model, and timestamp data - valuable forensic evidence.")
create_sample_image("sample.jpg")
show_metadata("sample.jpg")

File: sample.jpg
Format: JPEG
Size: (100, 100)
Mode: RGB
 No EXIF metadata found (this sample image has none).
 Real photos from phones/cameras typically contain GPS,
 device model, and timestamp data - valuable forensic evidence.


In [47]:
import datetime
def generate_forensic_report(case_id, findings, investigator):
    lines = []
    lines.append("DIGITAL FORENSIC INVESTIGATION REPORT")
    lines.append(f"Case ID: {case_id}")
    lines.append(f"Investigator: {investigator}")
    lines.append(f"Report generated: {datetime.datetime.now()}")
    lines.append("-" * 45)
    lines.append("FINDINGS:")
    for i, finding in enumerate(findings, 1):
        lines.append(f" {i}. {finding}")
    lines.append("-" * 45)
    lines.append("CONCLUSION:")
    lines.append(" Evidence indicates unauthorized access consistent with a")
    lines.append(" brute-force attack. Recommend IP block and password reset.")
    lines.append(" This report is suitable for legal review and expert testimony.")
    return "\n".join(lines)

findings = [
    "203.0.113.99 attempted 5 failed logins within 20 seconds (log file evidence).",
    "SHA-256 hash of evidence file verified unchanged throughout custody.",
    "Email header traced to a spoofed domain via suspicious relay server.",
]
report = generate_forensic_report("CASE-2026-014", findings, "Analyst B")
print(report)
with open("forensic_report.txt", "w") as f:
    f.write(report)

DIGITAL FORENSIC INVESTIGATION REPORT
Case ID: CASE-2026-014
Investigator: Analyst B
Report generated: 2026-08-04 04:14:54.629528
---------------------------------------------
FINDINGS:
 1. 203.0.113.99 attempted 5 failed logins within 20 seconds (log file evidence).
 2. SHA-256 hash of evidence file verified unchanged throughout custody.
 3. Email header traced to a spoofed domain via suspicious relay server.
---------------------------------------------
CONCLUSION:
 Evidence indicates unauthorized access consistent with a
 brute-force attack. Recommend IP block and password reset.
 This report is suitable for legal review and expert testimony.


In [48]:
import hashlib
import os
import requests
from google.colab import userdata
import time

# --- File Hash Generation ---

def generate_file_hashes(filepath):
    """Generates MD5, SHA-1, and SHA-256 hashes for a given file."""
    hashes = {
        'md5': hashlib.md5(),
        'sha1': hashlib.sha1(),
        'sha256': hashlib.sha256()
    }

    with open(filepath, 'rb') as f:
        while chunk := f.read(4096):
            for h in hashes.values():
                h.update(chunk)

    return {name: h.hexdigest() for name, h in hashes.items()}

# Create a dummy file for demonstration
sample_filename = 'sample_malware_file.txt'
with open(sample_filename, 'w') as f:
    f.write('This is a simulated malware file content. Do not run this on a real system.')
    f.write('Adding more content to make the file larger and more realistic for hashing.')

print(f"Created sample file: {sample_filename}")

# Generate and display hashes for the sample file
file_hashes = generate_file_hashes(sample_filename)
print("\nHashes for {}:".format(sample_filename))
for hash_type, hash_value in file_hashes.items():
    print(f"  {hash_type.upper()}: {hash_value}")

# os.remove(sample_filename) # Uncomment to clean up the sample file

# --- VirusTotal API Query ---

# Install the requests library if not already installed
!pip install requests --quiet

# Get the VirusTotal API key - User requested to hardcode it.
# IMPORTANT: Replace "YOUR_VIRUSTOTAL_API_KEY_HERE" with your actual VirusTotal API key.
# Hardcoding API keys is generally not recommended for security reasons.
VIRUSTOTAL_API_KEY = "YOUR_VIRUSTOTAL_API_KEY_HERE"

def query_virustotal(hash_value, api_key):
    """Queries the VirusTotal API for a given file hash."""
    url = f"https://www.virustotal.com/api/v3/files/{hash_value}"
    headers = {
        "x-apikeys": api_key,
        "Accept": "application/json"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
        return response.json()
    except requests.exceptions.HTTPError as http_err:
        if response.status_code == 404:
            print(f"  Hash not found on VirusTotal (404 Not Found).")
            return None
        elif response.status_code == 429:
            print(f"  Rate limit exceeded (429 Too Many Requests). Please wait and try again.")
            return None
        else:
            print(f"  HTTP error occurred: {http_err}")
            return None
    except Exception as err:
        print(f"  An error occurred: {err}")
        return None

print("\nQuerying VirusTotal for each generated hash...")

if not VIRUSTOTAL_API_KEY or VIRUSTOTAL_API_KEY == "71b4f79b55eeafbe1dac8ba2128a6ea059b3f7b11a933446d135f99f51bd815c":
    print("Error: VIRUSTOTAL_API_KEY is not set or is the placeholder. Please update it to proceed.")
else:
    for hash_type, hash_value in file_hashes.items():
        print(f"\nChecking {hash_type.upper()} hash: {hash_value}")
        vt_report = query_virustotal(hash_value, VIRUSTOTAL_API_KEY)

        if vt_report:
            data = vt_report.get('data')
            if data:
                attributes = data.get('attributes')
                if attributes:
                    last_analysis_stats = attributes.get('last_analysis_stats')
                    if last_analysis_stats:
                        harmless = last_analysis_stats.get('harmless', 0)
                        malicious = last_analysis_stats.get('malicious', 0)
                        undetected = last_analysis_stats.get('undetected', 0)

                        print(f"  VirusTotal Analysis Summary:")
                        print(f"    Malicious: {malicious}")
                        print(f"    Harmless: {harmless}")
                        print(f"    Undetected: {undetected}")
                        print(f"    Total engines: {harmless + malicious + undetected}")

                        if malicious > 0:
                            print(f"  * This hash is detected as MALICIOUS by {malicious} engines. *")
                        else:
                            print(f"  * This hash is not detected as malicious by any engine. *")
                    else:
                        print("  No last_analysis_stats found in report.")
                else:
                    print("  No attributes found in report data.")
            else:
                print("  No data found in VirusTotal report.")
        else:
            print(f"  Could not retrieve report for {hash_type.upper()} hash.")

        time.sleep(1) # Add a small delay to avoid hitting API rate limits


Created sample file: sample_malware_file.txt

Hashes for sample_malware_file.txt:
  MD5: 1bba55592c01c2843dbed1489145e308
  SHA1: b036ba271100f885bdb1c4ec7ce85b714855f024
  SHA256: 1816b15d13f7cc3045403fa4a74f01d178c9d4f2e18eff9fdb55f0cc335768c4

Querying VirusTotal for each generated hash...

Checking MD5 hash: 1bba55592c01c2843dbed1489145e308
  HTTP error occurred: 401 Client Error: Unauthorized for url: https://www.virustotal.com/api/v3/files/1bba55592c01c2843dbed1489145e308
  Could not retrieve report for MD5 hash.

Checking SHA1 hash: b036ba271100f885bdb1c4ec7ce85b714855f024
  HTTP error occurred: 401 Client Error: Unauthorized for url: https://www.virustotal.com/api/v3/files/b036ba271100f885bdb1c4ec7ce85b714855f024
  Could not retrieve report for SHA1 hash.

Checking SHA256 hash: 1816b15d13f7cc3045403fa4a74f01d178c9d4f2e18eff9fdb55f0cc335768c4
  HTTP error occurred: 401 Client Error: Unauthorized for url: https://www.virustotal.com/api/v3/files/1816b15d13f7cc3045403fa4a74f01d178c

In [49]:
import os
import hashlib

In [50]:
import os
evidence_dir = "digital_evidence"
os.makedirs(evidence_dir, exist_ok=True)

In [51]:
import os
import hashlib

evidence_dir = "digital_evidence"
os.makedirs(evidence_dir, exist_ok=True)
sample_evidence = {
    "email_log.txt": """From: attacker@mail.com\nTo: victim@mail.com\nSubject: Invoice\nPlease pay immediately.""",
    "browser_history.txt": "http://malicious-site.com/login\nhttp://bank.com/transfer",
    "chat_message.txt": "Hey, did you send the file?",
    "system_log.txt": "2026-07-24 10:15:32 - USB device connected: E:\\.",
}

for filename, content in sample_evidence.items():
    with open(os.path.join(evidence_dir, filename), "w") as f:
        f.write(content)

def catalog_evidence(folder):
    catalog = []
    for fname in sorted(os.listdir(folder)):
        path = os.path.join(folder, fname)
        stat = os.stat(path)
        with open(path, "rb") as f:
            data = f.read()
        catalog.append({
            "filename": fname,
            "size_bytes": stat.st_size,
            "md5": hashlib.md5(data).hexdigest(),
        })
    return catalog

evidence_catalog = catalog_evidence(evidence_dir)
for item in evidence_catalog:
    print(item)

{'filename': 'browser_history.txt', 'size_bytes': 56, 'md5': '7804eb386588e70e52f8e44985aae9ab'}
{'filename': 'chat_message.txt', 'size_bytes': 27, 'md5': 'e5f0758dbf1fe3c48005ea2560fa5279'}
{'filename': 'email_log.txt', 'size_bytes': 84, 'md5': '94d472f8d614b91afc7ab2f56ddb435d'}
{'filename': 'system_log.txt', 'size_bytes': 48, 'md5': 'c43bdc9f6e214a2e73df493bbe761ae7'}


In [52]:
!pip install psutil --quiet

In [53]:
import psutil
import os

In [54]:
import psutil
import os

def capture_volatile_evidence():
    processes = [p.info for p in psutil.process_iter(['pid', 'name'])]
    return {
        "running_process_count": len(processes),
        "sample_processes": processes[:5],
    }
def capture_nonvolatile_evidence(folder):
    files = sorted(os.listdir(folder))
    return {"disk_files": files, "file_count": len(files)}
volatile_snapshot = capture_volatile_evidence()
nonvolatile_snapshot = capture_nonvolatile_evidence(evidence_dir)
print("Volatile evidence -> running processes:",
volatile_snapshot["running_process_count"])
print("Non-volatile evidence -> disk files:", nonvolatile_snapshot)

Volatile evidence -> running processes: 15
Non-volatile evidence -> disk files: {'disk_files': ['browser_history.txt', 'chat_message.txt', 'email_log.txt', 'system_log.txt'], 'file_count': 4}


In [55]:
os.makedirs("acquisition_demo/source", exist_ok=True)
source = "acquisition_demo/source"
with open(os.path.join(source, "report.docx"), "w") as f:
    f.write("Confidential quarterly report.")
with open(os.path.join(source, "photo.jpg"), "w") as f:
    f.write("FAKEJPEGDATA")
raw_disk = b"REPORTDOCX_CONTENT" + b"\x00" * 20 + b"DELETED_INVOICE_DATA" + b"\x00" * 10
def physical_acquisition(raw_bytes):
    return bytes(raw_bytes) # every bit: used + unused + deleted
def logical_acquisition(folder):
    return {f: open(os.path.join(folder, f), "rb").read() for f in os.listdir(folder)}
def sparse_acquisition(folder, targets):
    return {f: open(os.path.join(folder, f), "rb").read() for f in targets if f in os.listdir(folder)}

physical_image = physical_acquisition(raw_disk)
logical_image = logical_acquisition(source)
sparse_image = sparse_acquisition(source, ["report.docx"])
print("Physical image size:", len(physical_image), "bytes")
print("Logical image files:", list(logical_image.keys()))
print("Sparse image files:", list(sparse_image.keys()))

Physical image size: 68 bytes
Logical image files: ['photo.jpg', 'report.docx']
Sparse image files: ['report.docx']


In [56]:
import time
import psutil
import json

In [57]:
def live_acquisition():
    return {
        "timestamp": time.time(),
        "running_processes": len(psutil.pids()),
        "cpu_percent": psutil.cpu_percent(interval=0.1),
    }
def dead_acquisition(snapshot_file):
    with open(snapshot_file) as f:
        return json.load(f)
dead_snapshot_path = "system_snapshot.json"
static_snapshot = {"hard_disk_files": ["a.txt", "b.txt"], "ram_data": None}
with open(dead_snapshot_path, "w") as f:
    json.dump(static_snapshot, f)
live_result = live_acquisition()
dead_result = dead_acquisition(dead_snapshot_path)
print("Live acquisition:", live_result)
print("Dead acquisition:", dead_result)

Live acquisition: {'timestamp': 1785816963.4114482, 'running_processes': 15, 'cpu_percent': 65.0}
Dead acquisition: {'hard_disk_files': ['a.txt', 'b.txt'], 'ram_data': None}


In [58]:
raw_disk_full = b"FILE1DATA" + b"\x00" * 15 + b"DELETED_FILE_DATA" + b"\x00" * 15 + b"FREE_SPACE_00000"
def create_forensic_image(raw_bytes):
    return bytes(raw_bytes) # bit-for-bit: files + deleted data + free space
def create_duplication(raw_bytes, active_regions):
    return b"".join(raw_bytes[start:end] for start, end in active_regions)
active_regions = [(0, 9)] # only the live "FILE1DATA" region
forensic_image = create_forensic_image(raw_disk_full)
duplication_copy = create_duplication(raw_disk_full, active_regions)
print("Forensic image size:", len(forensic_image))
print("Duplication size:", len(duplication_copy))

Forensic image size: 72
Duplication size: 9


In [59]:
import os
import hashlib

original_path = "original_evidence.bin"
with open(original_path, "wb") as f:
    f.write(os.urandom(1024)) # simulate a small storage device

def bit_stream_copy(src_path, dst_path):
    with open(src_path, "rb") as src, open(dst_path, "wb") as dst:
        dst.write(src.read())

copy_path = "bitstream_copy.bin"
bit_stream_copy(original_path, copy_path)

def sha256_of_file(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

original_hash = sha256_of_file(original_path)
copy_hash = sha256_of_file(copy_path)
print("Original hash:", original_hash)
print("Copy hash: ", copy_hash)

Original hash: 222854bb731cd2af2dd542f950d543df980836e930c6f5a1642a7454d87cebaf
Copy hash:  222854bb731cd2af2dd542f950d543df980836e930c6f5a1642a7454d87cebaf


In [60]:
import hashlib

def compute_hashes(data: bytes):
    return {
        "MD5": hashlib.md5(data).hexdigest(),
        "SHA1": hashlib.sha1(data).hexdigest(),
        "SHA256": hashlib.sha256(data).hexdigest(),
    }

original_data = b"hello"
tampered_data = b"Hello"
hashes_original = compute_hashes(original_data)
hashes_tampered = compute_hashes(tampered_data)
hashes_original_repeat = compute_hashes(original_data)
print("hello ->", hashes_original)
print("Hello ->", hashes_tampered)

hello -> {'MD5': '5d41402abc4b2a76b9719d911017c592', 'SHA1': 'aaf4c61ddcc5e8a2dabede0f3b482cd9aea9434d', 'SHA256': '2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824'}
Hello -> {'MD5': '8b1a9953c4611296a827abf8c47804d7', 'SHA1': 'f7ff9e8b7bb2e09b70935a5d785e0cc5d9d0abf0', 'SHA256': '185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969'}


In [61]:
import os

def make_fake_jpeg(payload: bytes):
    return b"\xff\xd8\xff" + payload + b"\xff\xd9"

jpeg1 = make_fake_jpeg(b"PHOTO_OF_SUSPECT_CAR")
jpeg2 = make_fake_jpeg(b"CCTV_FRAME_CAPTURE")
raw_disk_blob = os.urandom(30) + jpeg1 + os.urandom(40) + jpeg2 + os.urandom(20)

def carve_jpegs(blob: bytes):
    recovered = []
    start_marker, end_marker = b"\xff\xd8\xff", b"\xff\xd9"
    pos = 0
    while True:
        start = blob.find(start_marker, pos)
        if start == -1:
            break
        end = blob.find(end_marker, start)
        if end == -1:
            break
        end += len(end_marker)
        recovered.append(blob[start:end])
        pos = end
    return recovered

recovered_files = carve_jpegs(raw_disk_blob)

In [62]:
class FATFileEntry:
    def __init__(self, name, size):
        self.name = name
        self.size = size
        # FAT intentionally has no permissions or journal attribute
class NTFSFileEntry:
    def __init__(self, name, size, permissions="rw-r--r--"):
        self.name = name
        self.size = size
        self.permissions = permissions
        self.journal_entry = f"WRITE {name} {size} bytes"

class EXTFileEntry:
    def __init__(self, name, size, permissions="rw-r--r--"):
        self.name = name
        self.size = size
        self.permissions = permissions
        self.journal_entry = f"WRITE {name} {size} bytes"
fat_file = FATFileEntry("data.txt", 1024)
ntfs_file = NTFSFileEntry("data.txt", 1024)
ext_file = EXTFileEntry("data.txt", 1024)
print("FAT has permissions attribute:", hasattr(fat_file, "permissions"))
print("NTFS has permissions attribute:", hasattr(ntfs_file, "permissions"))
print("EXT has permissions attribute:", hasattr(ext_file, "permissions"))

FAT has permissions attribute: False
NTFS has permissions attribute: True
EXT has permissions attribute: True


In [63]:
CLUSTER_SIZE = 4096 # typical disk cluster size in bytes
def calculate_slack_space(file_size, cluster_size=CLUSTER_SIZE):
    clusters_needed = -(-file_size // cluster_size) # ceiling division
    allocated_space = clusters_needed * cluster_size
    slack_space = allocated_space - file_size
    return allocated_space, slack_space
file_size = 5000 # bytes
allocated, slack = calculate_slack_space(file_size)
print(f"File size: {file_size} bytes -> allocated: {allocated} bytes, slack space:\n{slack} bytes")

File size: 5000 bytes -> allocated: 8192 bytes, slack space:
3192 bytes


In [64]:
import hashlib
import json

def acquire(path):
  with open(path, "rb") as f:
    return f.read()

def hash_data(data):
  return hashlib.sha256(data).hexdigest()

def verify_integrity(data, expected_hash):
  return hash_data(data) == expected_hash

def analyze(data, keyword: bytes):
  return keyword in data

def generate_report(case_id, file_hash, keyword_found):
  return {
      "case_id": case_id,
      "sha256": file_hash,
      "suspicious_keyword_found": keyword_found,
      "status": "Evidence Verified" if keyword_found else "No Match",
  }

evidence_path = "workflow_evidence.txt"
with open(evidence_path, "w") as f:
  f.write("Transfer $50000 to account 99881122 immediately, do not report.")
acquired_data = acquire(evidence_path)
evidence_hash = hash_data(acquired_data)
integrity_ok = verify_integrity(acquired_data, evidence_hash)
keyword_found = analyze(acquired_data, b"99881122")
report = generate_report("CASE-2026-014", evidence_hash, keyword_found)
print(json.dumps(report, indent=2))

{
  "case_id": "CASE-2026-014",
  "sha256": "5761dda7d7476e1b92c42bcb9454f9c018f7b70adcac81879551e70c6f42f4a1",
  "suspicious_keyword_found": true,
  "status": "Evidence Verified"
}


In [65]:
from datetime import datetime, timedelta

def parse_time(t):
    return datetime.strptime(t, "%Y-%m-%d %H:%M:%S")

def detect_bruteforce(events, threshold=5, window_minutes=2):
    """Detect brute-force login attempts: >= threshold failed logons (4625)
    for the same account within window_minutes, and report whether a
    successful logon (4624) followed."""
    events = sorted(events, key=lambda e: parse_time(e["timestamp"]))
    by_account = {}
    for e in events:
        by_account.setdefault(e["account"], []).append(e)
    results = {}
    for account, acc_events in by_account.items():
        failures = [e for e in acc_events if e["event_id"] == 4625]
        successes = [e for e in acc_events if e["event_id"] == 4624]
        flagged = False
        for i in range(len(failures)):
            window_start = parse_time(failures[i]["timestamp"])
            window_end = window_start + timedelta(minutes=window_minutes)
            count = sum(
                1 for f in failures
                if window_start <= parse_time(f["timestamp"]) <= window_end
            )
            if count >= threshold:
                flagged = True
                break
        if flagged:
            followed_by_success = bool(successes) and any(
                parse_time(s["timestamp"]) > parse_time(failures[-1]["timestamp"])
                for s in successes
            )
            results[account] = {
                "failed_attempts": len(failures),
                "followed_by_success": followed_by_success,
                "source_ips": sorted({f["source_ip"] for f in failures}),
            }
    return results

In [66]:
from datetime import datetime, timedelta

def test_experiment1():
  events = []
  base = datetime(2026, 1, 15, 3, 40, 0)
  for i in range(6):
    events.append({
    "event_id": 4625, "account": "Administrator",
    "timestamp": (base + timedelta(seconds=15 * i)).strftime("%Y-%m-%d %H:%M:%S"),
    "source_ip": "203.0.113.7",
    })
  events.append({
    "event_id": 4624, "account": "Administrator",
    "timestamp": (base + timedelta(seconds=100)).strftime("%Y-%m-%d %H:%M:%S"),
    "source_ip": "203.0.113.7",
    })
  events.append({
    "event_id": 4624, "account": "jsmith",
    "timestamp": "2026-01-15 09:00:00", "source_ip": "10.0.0.5",
    })
  results = detect_bruteforce(events)
  assert "Administrator" in results
  assert results["Administrator"]["failed_attempts"] == 6
  assert results["Administrator"]["followed_by_success"] is True
  assert results["Administrator"]["source_ips"] == ["203.0.113.7"]
  assert "jsmith" not in results
  print("All test cases passed.")

test_experiment1()

All test cases passed.


In [67]:
from datetime import datetime

def parse_usbstor(registry_data):
    """registry_data: {device_id: {"serial", "friendly_name",
    "first_connected", "last_connected"}}. Returns list of device dicts
    sorted by last_connected, most recent first."""
    devices = []
    for device_id, info in registry_data.items():
        entry = {"device_id": device_id}
        entry.update(info)
        devices.append(entry)
    devices.sort(
        key=lambda d: datetime.strptime(d["last_connected"], "%Y-%m-%d %H:%M:%S"),
        reverse=True,
    )
    return devices

def find_device_near_time(devices, target_time_str, window_minutes=30):
    """Return devices whose last_connected time is within window_minutes
    of target_time_str."""
    target = datetime.strptime(target_time_str, "%Y-%m-%d %H:%M:%S")
    matches = []
    for d in devices:
        last = datetime.strptime(d["last_connected"], "%Y-%m-%d %H:%M:%S")
        delta_minutes = abs((target - last).total_seconds()) / 60
        if delta_minutes <= window_minutes:
            matches.append(d)
    return matches

In [68]:
from datetime import datetime

def test_experiment2():
  registry_data = {
  "USB\\VID_0781&PID_5567\\4C531001234": {
  "serial": "4C531001234",
  "friendly_name": "SanDisk Cruzer Blade",
  "first_connected": "2025-11-01 09:00:00",
  "last_connected": "2026-02-10 17:42:00",
  },
  "USB\\VID_090C&PID_1000\\A1002233": {
  "serial": "A1002233",
  "friendly_name": "Kingston DataTraveler",
  "first_connected": "2024-06-01 10:00:00",
  "last_connected": "2024-06-01 10:15:00",
  },
  }
  devices = parse_usbstor(registry_data)
  assert devices[0]["friendly_name"] == "SanDisk Cruzer Blade"
  matches = find_device_near_time(devices, "2026-02-10 17:35:00",
  window_minutes=30)
  matched_names = [m["friendly_name"] for m in matches]
  assert "SanDisk Cruzer Blade" in matched_names
  assert "Kingston DataTraveler" not in matched_names
  print("All test cases passed.")

test_experiment2()

All test cases passed.


In [69]:
import re
AUTH_LINE_RE = re.compile(
    r"(?P<result>Accepted|Failed) password for (?P<user>\S+) from (?P<ip>[\d.]+) port (?P<port>\d+)"
)

def parse_auth_log(lines):
    """Parse raw auth.log lines into structured dicts."""
    entries = []
    for line in lines:
        match = AUTH_LINE_RE.search(line)
        if match:
            entries.append({
                "result": match.group("result"),
                "user": match.group("user"),
                "ip": match.group("ip"),
                "port": int(match.group("port")),
                "raw": line,
            })
    return entries

def flag_suspicious_logins(entries, trusted_ips):
    """Flag successful logins from untrusted IPs, especially for root."""
    flagged = []
    for e in entries:
        if e["result"] == "Accepted" and e["ip"] not in trusted_ips:
            severity = "HIGH" if e["user"] == "root" else "MEDIUM"
            flagged.append({**e, "severity": severity})
    return flagged

In [70]:
def test_experiment3():
  log_lines = [
  "Jan 15 08:00:01 server sshd[1001]: Accepted password for deploy from 10.0.0.5 port 51100 ssh2",
  "Jan 15 03:12:01 server sshd[1233]: Failed password for root from 198.51.100.23 port 51320 ssh2",
  "Jan 15 03:12:05 server sshd[1234]: Accepted password for root from 198.51.100.23 port 51322 ssh2",
  ]
  trusted_ips = {"10.0.0.5"}
  entries = parse_auth_log(log_lines)
  assert len(entries) == 3
  flagged = flag_suspicious_logins(entries, trusted_ips)
  assert len(flagged) == 1
  assert flagged[0]["user"] == "root"
  assert flagged[0]["ip"] == "198.51.100.23"
  assert flagged[0]["severity"] == "HIGH"
  print("All test cases passed.")

test_experiment3()

All test cases passed.


In [71]:
from datetime import datetime
TS_FMT = "%Y-%m-%d %H:%M:%S"

def detect_timestomping(file_meta, change_gap_minutes=60):
    """file_meta: {"modified", "accessed", "changed", "born"} as timestamp
    strings. Returns (is_suspicious: bool, reasons: list[str])."""
    m = datetime.strptime(file_meta["modified"], TS_FMT)
    a = datetime.strptime(file_meta["accessed"], TS_FMT)
    c = datetime.strptime(file_meta["changed"], TS_FMT)
    b = datetime.strptime(file_meta["born"], TS_FMT)
    reasons = []
    if m < b:
        reasons.append("Modified time is earlier than Born (creation) time")
    if a < b:
        reasons.append("Accessed time is earlier than Born (creation) time")
    gap_minutes = abs((c - m).total_seconds()) / 60
    if gap_minutes > change_gap_minutes and c > m:
        reasons.append(
            f"MFT Changed time is {gap_minutes:.0f} minutes after Modified time — "
            "metadata may have been altered after the fact"
        )
    return (len(reasons) > 0, reasons)

In [72]:
from datetime import datetime

def test_experiment4():
  normal_file = {
  "born": "2026-01-10 09:00:00",
  "modified": "2026-01-10 09:05:00",
  "accessed": "2026-01-12 14:00:00",
  "changed": "2026-01-10 09:05:00",
  }
  tampered_file = {
  "born": "2026-02-01 12:00:00",
  "modified": "2020-01-01 00:00:00",
  "accessed": "2026-02-01 12:00:00",
  "changed": "2026-02-01 12:03:00",
  }
  is_susp1, reasons1 = detect_timestomping(normal_file)
  assert is_susp1 is False
  is_susp2, reasons2 = detect_timestomping(tampered_file)
  assert is_susp2 is True
  assert any("Modified time is earlier" in r for r in reasons2)
  print("All test cases passed.")

test_experiment4()

All test cases passed.


In [73]:
from datetime import datetime, timedelta
PKT_FMT = "%Y-%m-%d %H:%M:%S"

def detect_port_scan(packets, port_threshold=10, window_seconds=30):
    """Detect (src_ip -> dst_ip) pairs that contact >= port_threshold
    distinct destination ports within window_seconds."""
    packets = sorted(packets, key=lambda p: datetime.strptime(p["timestamp"],
    PKT_FMT))
    by_pair = {}
    for p in packets:
        key = (p["src_ip"], p["dst_ip"])
        by_pair.setdefault(key, []).append(p)
    results = {}
    for key, pkts in by_pair.items():
        for i in range(len(pkts)):
            start = datetime.strptime(pkts[i]["timestamp"], PKT_FMT)
            end = start + timedelta(seconds=window_seconds)
            ports_in_window = {
                p["dst_port"] for p in pkts
                if start <= datetime.strptime(p["timestamp"], PKT_FMT) <= end
            }
            if len(ports_in_window) >= port_threshold:
                results[key] = {
                    "distinct_ports": len(ports_in_window),
                    "ports": sorted(ports_in_window),

                }
                break
    return results

In [74]:
from datetime import datetime, timedelta

def test_experiment5():
  packets = []
  base = datetime(2026, 3, 1, 10, 0, 0)
  for i, port in enumerate(range(20, 32)):
    packets.append({
    "src_ip": "203.0.113.99", "dst_ip": "10.0.0.10",
    "dst_port": port,
    "timestamp": (base + timedelta(seconds=2 * i)).strftime(PKT_FMT),
    })
  for i in range(5):
    packets.append({
    "src_ip": "10.0.0.20", "dst_ip": "10.0.0.30",
    "dst_port": 443,
    "timestamp": (base + timedelta(seconds=5 * i)).strftime(PKT_FMT),
    })
  results = detect_port_scan(packets, port_threshold=10, window_seconds=30)
  assert ("203.0.113.99", "10.0.0.10") in results
  assert results[('203.0.113.99', '10.0.0.10')]["distinct_ports"] >= 10
  assert ("10.0.0.20", "10.0.0.30") not in results
  print("All test cases passed.")

test_experiment5()

All test cases passed.


In [75]:
import math
from collections import Counter

def shannon_entropy(s):
    if not s:
        return 0.0
    counts = Counter(s)
    length = len(s)
    return -sum((c / length) * math.log2(c / length) for c in counts.values())

def detect_dns_tunneling(queries, length_threshold=20, entropy_threshold=3.5):
    """queries: list of fully-qualified domain name strings.
    Flags queries whose leftmost label is long AND high-entropy."""
    flagged = []
    for q in queries:
        label = q.split(".")[0]
        entropy = shannon_entropy(label)
        if len(label) >= length_threshold and entropy >= entropy_threshold:
            flagged.append({"query": q, "label_length": len(label), "entropy":
            round(entropy, 2)})
    return flagged

In [76]:
def test_experiment6():
  queries = [
  "www.google.com",
  "mail.office365.com",
  "a8f3k2j9x1p7q4z6w0n5r2t8y3.exfil-domain.com",
  "vpn.corporate-network.com",

  "9c2e7b1a4f8d3c6e0a5b9d2f7c1e4a8b3d6f9c2e5a8b1d4f.tunnel.example.net",
  ]
  flagged = detect_dns_tunneling(queries)
  flagged_domains = [f["query"] for f in flagged]
  assert "www.google.com" not in flagged_domains
  assert "mail.office365.com" not in flagged_domains
  assert "vpn.corporate-network.com" not in flagged_domains
  assert "a8f3k2j9x1p7q4z6w0n5r2t8y3.exfil-domain.com" in flagged_domains
  assert ("9c2e7b1a4f8d3c6e0a5b9d2f7c1e4a8b3d6f9c2e5a8b1d4f.tunnel.example.net"
          in flagged_domains)
  assert len(flagged) == 2
  print("All test cases passed.")

test_experiment6()

All test cases passed.


In [77]:
SIGNATURES = [
  {"sid": 1000001, "name": "Possible SQL Injection", "pattern": "union select"},
  {"sid": 1000002, "name": "Directory Traversal Attempt", "pattern":
  "../../../etc/passwd"},
  {"sid": 1000003, "name": "Nmap Scripting Engine User-Agent", "pattern": "nmap scripting engine"},
  ]

def scan_payloads(packets, signatures=SIGNATURES):
  """packets: list of dicts with a 'payload' string field.
  Returns list of alerts: {packet_index, sid, name}."""
  alerts = []
  for idx, pkt in enumerate(packets):
    payload_lower = pkt["payload"].lower()
    for sig in signatures:
      if sig["pattern"] in payload_lower:
        alerts.append({
          "packet_index": idx,
          "sid": sig["sid"],
          "name": sig["name"],
        })
  return alerts

In [78]:
def test_experiment7():
  packets = [
  {"payload": "GET /products?id=1 HTTP/1.1"},
  {"payload": "GET /login?user=admin' UNION SELECT username,password FROM users-- HTTP/1.1"},
  {"payload": "GET /download?file=../../../etc/passwd HTTP/1.1"},
  ]

  alerts = scan_payloads(packets)
  alert_indices = {a["packet_index"] for a in alerts}
  assert 0 not in alert_indices
  assert 1 in alert_indices
  assert 2 in alert_indices
  assert any(a["name"] == "Possible SQL Injection" for a in alerts)
  assert any(a["name"] == "Directory Traversal Attempt" for a in alerts)
  print("All test cases passed.")

  test_experiment7()

In [79]:
def recover_deleted_messages(sms_table):
  """sms_table: list of dicts with is_deleted (bool) and overwritten (bool).
  Returns a list of rows that are deleted but still recoverable."""
  recoverable = [
  row for row in sms_table
  if row["is_deleted"] and not row["overwritten"]
  ]
  return recoverable

def summarize_table(sms_table):
  active = [r for r in sms_table if not r["is_deleted"]]
  recoverable = recover_deleted_messages(sms_table)
  lost = [r for r in sms_table if r["is_deleted"] and r["overwritten"]]
  return {"active": len(active), "recoverable": len(recoverable),
  "permanently_lost": len(lost)}

In [80]:
def test_experiment8():
  sms_table = [
  {"rowid": 1, "address": "+1-555-0101", "body": "See you at 6pm", "date":
  "2026-01-01", "is_deleted": False, "overwritten": False},
  {"rowid": 2, "address": "+1-555-0199", "body": "Transfer the funds now, delete after reading", "date": "2026-01-02", "is_deleted": True, "overwritten":
  False},
  {"rowid": 3, "address": "+1-555-0150", "body": "Old spam message", "date":
  "2025-06-01", "is_deleted": True, "overwritten": True},
  ]
  recoverable = recover_deleted_messages(sms_table)
  assert len(recoverable) == 1

  assert recoverable[0]["rowid"] == 2
  assert "Transfer the funds" in recoverable[0]["body"]
  summary = summarize_table(sms_table)
  assert summary == {"active": 1, "recoverable": 1, "permanently_lost": 1}
  print("All test cases passed.")

  test_experiment8()

In [81]:
from datetime import datetime
from collections import Counter
LOG_FMT = "%Y-%m-%d %H:%M:%S"

def build_baseline_ips(logs):
  """Return {user: most_common_ip} based on all log activity."""
  by_user = {}
  for entry in logs:
    by_user.setdefault(entry["user"], []).append(entry["ip"])
  return {user: Counter(ips).most_common(1)[0][0] for user, ips in
  by_user.items()}

def flag_anomalous_downloads(logs, business_start=8, business_end=20):
  baseline = build_baseline_ips(logs)
  flagged = []
  for entry in logs:
    if entry["action"] != "download":
      continue
    ts = datetime.strptime(entry["timestamp"], LOG_FMT)
    reasons = []
    if entry["ip"] != baseline.get(entry["user"]):
      reasons.append("IP differs from user baseline")
    if not (business_start <= ts.hour < business_end):
      reasons.append("Outside business hours")
    if reasons:
      flagged.append({**entry, "reasons": reasons})
  return flagged

In [82]:
def test_experiment9():
  logs = [
  {"user": "alice", "action": "view", "file": "roadmap.docx", "timestamp":
  "2026-03-01 10:00:00", "ip": "10.0.0.5"},
  {"user": "alice", "action": "view", "file": "budget.xlsx", "timestamp":
  "2026-03-02 11:00:00", "ip": "10.0.0.5"},
  {"user": "alice", "action": "download", "file": "report.pdf", "timestamp":
  "2026-03-03 14:00:00", "ip": "10.0.0.5"},
  {"user": "alice", "action": "download", "file": "customer_database.csv",
  "timestamp": "2026-03-05 02:15:00", "ip": "185.220.101.7"},
  ]
  flagged = flag_anomalous_downloads(logs)
  assert len(flagged) == 1
  assert flagged[0]["file"] == "customer_database.csv"
  assert "IP differs from user baseline" in flagged[0]["reasons"]
  assert "Outside business hours" in flagged[0]["reasons"]
  print("All test cases passed.")

  test_experiment9()

In [84]:
import hashlib
import math
from collections import Counter

def sha256_hash(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def byte_entropy(data: bytes) -> float:
    if not data:
        return 0.0
    counts = Counter(data)
    length = len(data)
    return -sum((c / length) * math.log2(c / length) for c in counts.values())

def static_analyze(data: bytes, entropy_threshold=7.5):
    return {
        "sha256": sha256_hash(data),
        "size_bytes": len(data),
        "entropy": round(byte_entropy(data), 2),
        "likely_packed": byte_entropy(data) >= entropy_threshold,
    }

In [86]:
import os
def test_experiment10():
    plain_sample = (b"This is a normal configuration file. " * 20)
    random_sample = os.urandom(2000)
    plain_result = static_analyze(plain_sample)
    random_result = static_analyze(random_sample)
    assert len(plain_result["sha256"]) == 64
    assert plain_result["entropy"] < 5.0
    assert plain_result["likely_packed"] is False
    assert random_result["entropy"] >= 7.5
    assert random_result["likely_packed"] is True
    assert sha256_hash(plain_sample) == sha256_hash(plain_sample)
    print("All test cases passed.")

test_experiment10()

All test cases passed.
